# B0 — Bayesian Modeling Primer *(optional)*

**Skip this notebook** if you already use PyMC or have a solid Bayesian modeling background.

**You need this notebook if:** you haven't seen Bayesian inference in practice, or
you've heard of Bayes' theorem but never written a probabilistic model in code.

**What you'll learn:**
- Prior, likelihood, posterior — what each term means and how to compute them
- MCMC intuition — what the sampler does and how to read diagnostics
- The PyMC workflow you'll use in B5 (parameter estimation)

## 1. Bayes' theorem

$$P(\theta \mid y) = \frac{P(y \mid \theta)\, P(\theta)}{P(y)} \propto P(y \mid \theta)\, P(\theta)$$

- **Prior** $P(\theta)$: what we believe about $\theta$ before seeing data
- **Likelihood** $P(y \mid \theta)$: how probable the data is given $\theta$
- **Posterior** $P(\theta \mid y)$: updated belief after seeing data

### Example: coin-flip

We observe $k$ heads in $n$ flips. We want to infer $\theta$ = P(heads).

Prior: $\theta \sim \text{Beta}(\alpha_0, \beta_0)$ (encodes prior belief)
Likelihood: $k \mid \theta \sim \text{Binomial}(n, \theta)$
Posterior: $\theta \mid k \sim \text{Beta}(\alpha_0 + k,\, \beta_0 + n - k)$ (conjugate)

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist, norm

# Data
n_flips, k_heads = 20, 14

# Prior: Beta(2, 2) — slight preference for fair coin
alpha0, beta0 = 2.0, 2.0

# Grid approximation of posterior
theta_grid = np.linspace(0.001, 0.999, 500)
prior = beta_dist.pdf(theta_grid, alpha0, beta0)
likelihood = theta_grid**k_heads * (1 - theta_grid)**(n_flips - k_heads)
posterior_unnorm = prior * likelihood
posterior = posterior_unnorm / np.trapz(posterior_unnorm, theta_grid)

# Exact conjugate posterior
alpha_post = alpha0 + k_heads
beta_post  = beta0 + (n_flips - k_heads)
posterior_exact = beta_dist.pdf(theta_grid, alpha_post, beta_post)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(theta_grid, prior / np.trapz(prior, theta_grid), "C0--", lw=1.5, label=f"Prior Beta({alpha0},{beta0})")
ax.plot(theta_grid, posterior, "C1-", lw=2, label="Grid posterior")
ax.plot(theta_grid, posterior_exact, "k:", lw=1.5, label=f"Conjugate Beta({alpha_post},{beta_post})")
ax.axvline(k_heads/n_flips, color="grey", ls="--", lw=1, label=f"MLE = {k_heads/n_flips:.2f}")
ax.set(xlabel="θ = P(heads)", ylabel="density", title=f"Coin flip: {k_heads}/{n_flips} heads")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Posterior mean (conjugate): {alpha_post/(alpha_post+beta_post):.3f}")

## 2. PyMC — same model in code

PyMC lets us write the generative model symbolically. MCMC then samples from the
posterior numerically — useful when there's no conjugate formula.

In [ ]:
HAS_PYMC = False
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
except ImportError:
    print("PyMC not installed — install with: pip install pymc arviz")

if HAS_PYMC:
    with pm.Model() as coin_model:
        theta = pm.Beta("theta", alpha=alpha0, beta=beta0)
        obs = pm.Binomial("obs", n=n_flips, p=theta, observed=k_heads)
        idata = pm.sample(2000, tune=1000, progressbar=False, random_seed=42)

    fig, ax = plt.subplots(figsize=(9, 4))
    samples = idata.posterior["theta"].values.ravel()
    ax.hist(samples, bins=60, density=True, alpha=0.5, color="C1", label="MCMC posterior")
    ax.plot(theta_grid, posterior_exact, "k-", lw=2, label="Conjugate posterior")
    ax.set(xlabel="θ", ylabel="density", title="PyMC vs conjugate posterior")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(az.summary(idata, var_names=["theta"]))

## 3. Gaussian mean estimation

A more DLM-relevant example: estimate the mean $\mu$ of a Gaussian with known variance.

**Model:**
$$\mu \sim N(\mu_0, \sigma_0^2), \quad y_i \mid \mu \sim N(\mu, \sigma^2)$$

**Conjugate posterior:**
$$\mu \mid y_{1:N} \sim N(\mu_N, \sigma_N^2)$$
where
$$\sigma_N^2 = \left(\frac{1}{\sigma_0^2} + \frac{N}{\sigma^2}\right)^{-1}, \quad
\mu_N = \sigma_N^2 \left(\frac{\mu_0}{\sigma_0^2} + \frac{\sum_i y_i}{\sigma^2}\right)$$

In [ ]:
rng = np.random.default_rng(0)
mu_true = 3.5
sigma = 1.0       # known observation std
N = 30
y_data = rng.normal(mu_true, sigma, N)

# Prior
mu0, sigma0 = 0.0, 5.0

# Conjugate posterior
sigma_N2 = 1.0 / (1/sigma0**2 + N/sigma**2)
mu_N = sigma_N2 * (mu0/sigma0**2 + y_data.sum()/sigma**2)

mu_grid = np.linspace(-2, 8, 400)
prior_pdf = norm.pdf(mu_grid, mu0, sigma0)
post_pdf  = norm.pdf(mu_grid, mu_N, np.sqrt(sigma_N2))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mu_grid, prior_pdf, "C0--", lw=1.5, label=f"Prior N({mu0},{sigma0}²)")
ax.plot(mu_grid, post_pdf,  "C1-",  lw=2,   label=f"Posterior N({mu_N:.2f},{sigma_N2:.3f})")
ax.axvline(mu_true, color="k", ls=":", lw=1.5, label=f"True μ = {mu_true}")
ax.axvline(y_data.mean(), color="grey", ls="--", lw=1, label=f"Sample mean = {y_data.mean():.2f}")
ax.set(xlabel="μ", ylabel="density", title="Gaussian mean estimation")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Posterior mean: {mu_N:.3f}  (true: {mu_true})")
print(f"Posterior std : {np.sqrt(sigma_N2):.3f}")

## 4. Reading MCMC diagnostics

When using MCMC (as in B5), check two diagnostics before trusting results:

| Diagnostic | Good value | Meaning |
|------------|-----------|---------|
| `r_hat` | < 1.01 | chains converged to same distribution |
| `ess_bulk` | > 400 | effective sample size — enough independent draws |

`az.plot_trace(idata)` shows: left column = density of samples, right column = trace over iterations.
A healthy trace looks like white noise (no trends, no stuck periods).

In [ ]:
if HAS_PYMC:
    with pm.Model() as gauss_model:
        mu = pm.Normal("mu", mu=mu0, sigma=sigma0)
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y_data)
        idata2 = pm.sample(2000, tune=1000, progressbar=False, random_seed=1)

    az.plot_trace(idata2, var_names=["mu"])
    plt.suptitle("MCMC trace and posterior — Gaussian mean")
    plt.tight_layout()
    plt.show()
    print(az.summary(idata2, var_names=["mu"]))
    print(f"\nConjugate posterior mean: {mu_N:.3f}")

## Exercises

**Exercise 1** — Prior sensitivity: change the prior to `Normal(mu=10, sigma=0.5)`
(strong prior far from true value) and re-run with N=5, then N=50. At which N does
the posterior "overcome" the prior?

**Exercise 2** — Posterior width formula: verify numerically that
$\sigma_N^2 = (1/\sigma_0^2 + N/\sigma^2)^{-1}$ matches `az.summary`'s `sd` column
(within floating-point tolerance) for N=30, σ=1, σ₀=5.